# Confronto delle colonne per le cancellazioni hotel
Analisi separata: il notebook esercizio_hotel.ipynb rimane invariato. Si replica il campione di 12.000 righe, lo split 80/20 e la cross-validation stratificata a 5 fold con random_state=42. Il test resta escluso dalla scelta. Obiettivo: migliorare F1 della classe cancellata mantenendo visibili precision e recall. Le aggiunte sono prima provate una alla volta, mantenendo la stessa Random Forest.
I dati originali contengono valori aggiornati fino a prima di arrivo/cancellazione: un vantaggio predittivo non garantisce che la colonna sia disponibile alla creazione della prenotazione. Fonte: https://repositorio.iscte-iul.pt/bitstream/10071/16929/1/Hotelbookingdemanddatasets.pdf

## Come orientarsi nel notebook

Questo notebook cerca una scelta di colonne utile. Per farlo ripete molti addestramenti: è il motivo per cui il codice è più articolato dell'esercizio.

1. **Preparazione:** leggiamo il CSV, riserviamo il test e definiamo due funzioni riutilizzabili.
2. **Prove singole:** partiamo dalle colonne iniziali e aggiungiamo oppure togliamo una colonna alla volta.
3. **Combinazioni:** proviamo insieme le aggiunte promettenti e verifichiamo il risultato anche con la regressione logistica.
4. **Controllo per gruppi:** ripetiamo il confronto tenendo i profili identici nello stesso fold, per evitare che attraversino training e validazione.
5. **Interpretazione:** leggiamo i risultati e ricaviamo le liste di colonne da usare nell'esercizio.

| Nome nel codice | A cosa serve |
|---|---|
| `crea_modello(...)` | Costruisce una pipeline da addestrare. La sola chiamata non avvia l'apprendimento. |
| `valuta(...)` | Richiama la prima funzione, esegue cinque addestramenti di cross-validation e registra i punteggi. |
| `risultati_analisi` | Conserva le righe riassuntive, con le medie delle metriche. |
| `punteggi_analisi` | Conserva anche i cinque valori separati di ogni metrica, per confrontare gli stessi fold. |

**Ordine di esecuzione:** parti dalla prima cella di codice e procedi verso il basso. La prima cella azzera i contenitori dei risultati. Rieseguire soltanto le celle delle prove può accumulare righe ripetute: i commenti spiegano dove e perché.

**Come leggere i punteggi:** in questo notebook la classe positiva è la prenotazione cancellata. Il criterio di confronto è F1, affiancato da precision, recall, accuracy e AUC. Le chiavi come `test_f1` restituite da scikit-learn indicano la validazione del singolo fold; il test finale riservato all'inizio resta escluso da questi confronti.


In [5]:
# COME LEGGERE ED ESEGUIRE QUESTO NOTEBOOK
# Lo scopo è capire quali colonne aiutano a prevedere la cancellazione di una prenotazione.
# Esegui le celle dall'alto verso il basso: le variabili e le funzioni create qui serviranno dopo.
# Il kernel è il processo che esegue Python e conserva questi oggetti in memoria.
# Le righe che iniziano con # sono spiegazioni: Python non le esegue.
# Questa cella carica i dati e definisce gli strumenti; le prove vere saranno avviate dalle celle successive.
#
# 1. LIBRERIE: GLI STRUMENTI CHE USEREMO
# 'import' rende disponibili funzioni scritte da altri. 'as' assegna un'abbreviazione.
# Path gestisce i percorsi dei file senza dover concatenare manualmente separatori.
from pathlib import Path
# time permette di misurare quanti secondi richiede ogni esperimento.
import time
# NumPy, abbreviato np, gestisce vettori di numeri: servirà per confrontare i punteggi dei fold.
import numpy as np
# pandas, abbreviato pd, gestisce tabelle chiamate DataFrame.
# Ogni riga descrive una prenotazione; ogni colonna contiene una sua caratteristica.
import pandas as pd
# scikit-learn contiene modelli e strumenti di machine learning, cioè di apprendimento dai dati.
# clone crea una copia non addestrata di un modello con le stesse impostazioni;
# qui viene importato, ma il codice di questo notebook non lo richiama direttamente.
from sklearn.base import clone
# ColumnTransformer applica trasformazioni diverse a elenchi diversi di colonne.
from sklearn.compose import ColumnTransformer
# Una Random Forest combina molti alberi decisionali, ciascuno formato da domande sulle caratteristiche.
# La foresta mette insieme le stime degli alberi per classificare una nuova prenotazione.
from sklearn.ensemble import RandomForestClassifier
# SimpleImputer sostituisce eventuali dati mancanti usando una regola appresa nel training.
from sklearn.impute import SimpleImputer
# La regressione logistica combina le caratteristiche per stimare una probabilità.
# Qui il risultato da classificare ha due valori: 0 = non cancellata, 1 = cancellata.
from sklearn.linear_model import LogisticRegression
# train_test_split separa training e test. StratifiedKFold suddivide il training in parti.
# cross_validate addestra e valuta copie del modello su queste diverse suddivisioni.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
# Una Pipeline è una sequenza ordinata: prima prepara i dati, poi applica il classificatore.
from sklearn.pipeline import Pipeline
# StandardScaler rende confrontabili le scale numeriche. OneHotEncoder trasforma le categorie in indicatori 0/1.
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 2. CARICAMENTO E CAMPIONAMENTO
# Il segno = assegna un valore a un nome: potremo riutilizzare quel valore tramite il nome.
# Questo è il percorso assoluto della cartella che contiene hotels.csv.
cartella = Path('C:/Users/corra/Documents/GitHub/ai-specialist-python-lessons/14_classificazione')
# Con un oggetto Path, / unisce cartella e nome del file: qui non indica una divisione.
# read_csv legge il CSV, un file di testo organizzato in righe e colonne, e crea la tabella completo.
completo = pd.read_csv(cartella / 'hotels.csv')
# sample sceglie righe casuali, senza cambiare i valori delle prenotazioni.
# len(completo) conta le righe; min sceglie il minore tra 12.000 e le righe disponibili.
# random_state=42 fissa la casualità per poter ripetere la stessa estrazione sugli stessi dati.
# reset_index rinumera le righe da zero; drop=True evita di conservare i vecchi numeri in una colonna.
# I punti collegano operazioni successive: prima estraiamo le righe, poi sistemiamo il loro indice.
campione = completo.sample(n=min(12000, len(completo)), random_state=42).reset_index(drop=True)
# 3. IDENTIFICATIVI NUMERICI CHE RAPPRESENTANO CATEGORIE
# agent e company contengono codici di agenzie e aziende, non importi o quantità.
# Per esempio, il codice 250 non significa 'il doppio' del codice 125.
# Le parentesi quadre creano una lista; il ciclo for ripete l'operazione per ciascun nome della lista.
# Gli identificativi non sono quantita. Il valore mancante significa non applicabile.
for col in ['agent', 'company']:
    # campione[col] seleziona la colonna il cui nome è attualmente contenuto nella variabile col.
    # astype cambia il tipo con cui sono rappresentati i valori, in quattro passaggi:
    # 1) 'Int64' indica interi pandas che possono contenere anche mancanti: ad esempio 250.0 diventa 250;
    # 2) 'string' traduce i codici in testo, rendendo esplicito che sono etichette;
    # 3) fillna sostituisce i mancanti con una categoria chiamata 'non_applicabile';
    # 4) object conserva i testi in un formato utilizzabile dalla pipeline categorica.
    # Questa conversione applica regole fisse: non stima medie o frequenze usando l'intero campione.
    campione[col] = campione[col].astype('Int64').astype('string').fillna('non_applicabile').astype(object)
# 4. COLONNE DELLA CONFIGURAZIONE DI PARTENZA, DETTA BASELINE
# Una feature è una caratteristica fornita al modello; il target è la risposta da prevedere.
# Le liste contengono i nomi delle colonne, non i valori delle prenotazioni.
# Le numeriche di partenza descrivono anticipo in giorni, notti, persone e precedenti prenotazioni.
numeriche_base = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults',
                  'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled']
# Le categoriche descrivono tipo di hotel, mese, pasti, paese, segmento, camera e tipo di cliente.
# Una categoria è un'etichetta: non va interpretata automaticamente come una quantità ordinata.
# 'Base' indica la selezione iniziale a cui aggiungeremo o toglieremo colonne nelle prove.
categoriche_base = ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
                    'reserved_room_type', 'customer_type']
# 5. TRAINING E TEST: RUOLI DIVERSI
# Il training serve a imparare e confrontare configurazioni; il test serve alla valutazione finale.
# Qui dividiamo le etichette delle righe (campione.index), invece di dividere subito tutte le colonne.
# La funzione restituisce due gruppi, assegnati ai due nomi a sinistra del segno =.
# test_size=0.20 riserva il 20% al test e lascia l'80% al training.
# stratify mantiene proporzioni simili di cancellazioni e non cancellazioni nei due gruppi.
# random_state rende riproducibile lo split, cioè la divisione, sugli stessi dati.
# Gli indici del test vengono conservati, ma in questo notebook non calcoliamo punteggi su quelle righe.
indici_train, indici_test = train_test_split(campione.index, test_size=0.20,
                                           random_state=42, stratify=campione['is_canceled'])
# loc seleziona righe tramite le loro etichette; copy crea una tabella separata per il training.
# train conserva tutte le colonne: ogni esperimento selezionerà soltanto quelle necessarie.
train = campione.loc[indici_train].copy()
# Con una singola colonna otteniamo una Series, cioè una sequenza di valori con indice.
# Questa contiene gli esiti reali delle righe di train: 0 = non cancellata, 1 = cancellata.
# Il target non verrà inserito tra le colonne date al classificatore.
y_train_analisi = train['is_canceled']
# 6. CROSS-VALIDATION: CINQUE PROVE SUL SOLO TRAINING
# Dividiamo train in 5 parti, chiamate fold: a turno si impara da 4 parti e si valuta sulla quinta.
# Ogni riga viene usata una volta per la validazione e nelle altre prove per l'addestramento.
# Stratified mantiene proporzioni simili delle classi nei fold; shuffle mescola prima le righe.
# split produce coppie di indici posizionali: righe per imparare e righe da tenere fuori in quel turno.
# list memorizza tutte le coppie, così ogni variante usa esattamente le stesse suddivisioni.
# Questi indici si riferiscono alle posizioni dentro train, non alle etichette originarie del CSV.
# Tenere fissi i fold permette un confronto più leggibile tra varianti.
fold_analisi = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, y_train_analisi))
# 7. METRICHE: CHE COSA SIGNIFICA OTTENERE UN BUON RISULTATO?
# Un dizionario associa chiavi a valori con la sintassi {chiave: valore}.
# Qui le chiavi sono nomi che useremo nell'output, i valori identificano i calcoli di scikit-learn.
# Accuracy: quota di prenotazioni classificate correttamente, considerando entrambe le classi.
# Precision: tra le previste cancellate, quota che è davvero cancellata.
# Recall: tra tutte le vere cancellazioni, quota che riusciamo a trovare.
# F1: media armonica di precision e recall; penalizza il caso in cui una delle due è bassa.
# Precision, recall e F1 qui si riferiscono alla classe positiva 1, cioè 'Cancellata'.
# ROC AUC: misura quanto il modello ordina le cancellazioni sopra le non cancellazioni nei suoi punteggi;
# 0,5 indica un ordinamento casuale e 1 perfetto. Non è la quota di classificazioni corrette.
# Il criterio di scelta di questo confronto è F1; osserviamo anche gli altri valori per capire gli errori.
metriche_analisi = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall',
                   'f1': 'f1', 'roc_auc': 'roc_auc'}
# 8. FUNZIONE crea_modello: COSTRUIRE UNA PIPELINE, SENZA ADDESTRARLA
# 'def' definisce una funzione: un blocco riutilizzabile che verrà eseguito quando lo richiamiamo.
# I parametri tra parentesi sono ingressi: qui due liste di colonne e, facoltativamente, un classificatore.
# None significa 'nessun valore fornito': se omettiamo il terzo ingresso useremo una Random Forest.
# Le righe rientrate appartengono alla funzione; quelle non rientrate dopo la funzione sono esterne.
# Dentro la funzione, numeriche e categoriche indicano le liste ricevute per quella specifica prova.
def crea_modello(numeriche, categoriche, classificatore=None):
    # Prepariamo i dati con regole diverse per i due gruppi di colonne.
    # Ogni tupla ha tre elementi: nome del gruppo, trasformazione da applicare, elenco delle colonne.
    # Le tuple sono raggruppamenti ordinati di valori, scritti tra parentesi tonde.
    # La prima Pipeline riempie i mancanti numerici con la media, poi standardizza la scala.
    # StandardScaler sottrae la media e divide per la deviazione standard calcolate sul training.
    # La seconda Pipeline riempie i mancanti categorici con la categoria più frequente, poi crea indicatori 0/1.
    # Per esempio, due tipi di hotel diventano due colonne: una vale 1 per City Hotel, l'altra per Resort Hotel.
    # handle_unknown='ignore' gestisce categorie nuove in validazione con indicatori tutti a zero.
    # ColumnTransformer unisce infine le colonne trasformate in un'unica matrice di numeri.
    # In questo punto descriviamo i passaggi: medie, scale e categorie non sono ancora state imparate.
    pre = ColumnTransformer([
        ('numeriche', Pipeline([('imputer', SimpleImputer(strategy='mean')),
                                ('scaler', StandardScaler())]), numeriche),
        ('categoriche', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categoriche)
    ])
    # if esegue il blocco successivo soltanto se la condizione è vera.
    # Se non abbiamo passato un classificatore, costruiamo quello predefinito con impostazioni fisse.
    # n_estimators=300 crea 300 alberi; min_samples_leaf=2 richiede almeno due esempi in ogni foglia.
    # Una foglia è il gruppo finale raggiunto seguendo le domande di un albero.
    # random_state fissa la casualità; n_jobs=-1 usa i processori disponibili per elaborare gli alberi.
    # Se invece abbiamo passato LogisticRegression(...), questo blocco viene saltato.
    if classificatore is None:
        classificatore = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                                random_state=42, n_jobs=-1)
    # return restituisce al chiamante la pipeline completa: preparazione seguita dal classificatore.
    # Ogni richiamo della funzione crea una nuova pipeline, con le colonne richieste per quella prova.
    # La standardizzazione è utile alla logistica e non è necessaria alla foresta: la conserviamo
    # per mantenere lo stesso schema di preprocessing nel confronto tra classificatori.
    # Nessun fit viene eseguito qui: il valore restituito è un modello da addestrare.
    # Durante la cross-validation, la pipeline imparerà il preprocessing solo dal training di ogni fold,
    # e lo applicherà alle righe di validazione senza imparare da esse: questo evita una fuga di informazioni.
    return Pipeline([('preprocessing', pre), ('classificatore', classificatore)])

# 9. CONTENITORI DEI RISULTATI
# [] crea una lista vuota: raccoglierà una riga riassuntiva per ogni esperimento.
# {} crea un dizionario vuoto: conserverà i punteggi dettagliati, associati al nome della variante.
# Queste variabili sono esterne alle funzioni e restano disponibili alle celle successive.
# Rieseguire questa cella le azzera; rieseguire soltanto le prove aggiunge altre righe alla lista.
risultati_analisi = []
punteggi_analisi = {}
# 10. FUNZIONE valuta: ESEGUIRE UN ESPERIMENTO E REGISTRARNE I RISULTATI
# nome è un'etichetta descrittiva, non una colonna del CSV né un'impostazione del modello.
# Le liste numeriche e categoriche stabiliscono quali caratteristiche provare questa volta.
# Anche qui classificatore=None fa usare la Random Forest predefinita.
# Questa funzione usa oggetti creati fuori: train, y_train_analisi, fold_analisi e metriche_analisi.
# Le funzioni, da sole, non avviano le prove: occorre chiamare valuta(...) nelle celle successive.
def valuta(nome, numeriche, categoriche, classificatore=None):
    # Registriamo l'istante iniziale con un contatore preciso; sottraendolo all'istante finale avremo i secondi trascorsi.
    inizio = time.perf_counter()
    # Prima crea_modello costruisce la pipeline richiesta, poi cross_validate ne addestra copie indipendenti.
    # train[numeriche + categoriche] seleziona le colonne: + concatena due liste di nomi.
    # y_train_analisi fornisce gli esiti reali delle stesse righe.
    # cv=fold_analisi riutilizza i cinque split già preparati; scoring richiede tutte le metriche indicate.
    # n_jobs=1 esegue i fold uno alla volta; la Random Forest può comunque parallelizzare i suoi alberi.
    # error_score='raise' rende visibile un errore di addestramento invece di assegnare un punteggio mancante.
    # Per ogni variante vengono quindi eseguiti cinque addestramenti, sul solo training disponibile.
    # score è un dizionario: per esempio score['test_f1'] è un vettore di cinque F1, uno per fold.
    # In questi nomi, 'test_' significa la parte lasciata fuori nel fold, non il test finale riservato.
    score = cross_validate(crea_modello(numeriche, categoriche, classificatore),
                           train[numeriche + categoriche], y_train_analisi,
                           cv=fold_analisi, scoring=metriche_analisi, n_jobs=1, error_score='raise')
    # Conserviamo i punteggi grezzi per poter confrontare in seguito le differenze tra gli stessi fold.
    # Se un nome esiste già nel dizionario, questa assegnazione ne sostituisce il valore.
    punteggi_analisi[nome] = score
    # Creiamo il dizionario che rappresenterà una riga della tabella finale.
    riga = {'variante': nome}
    # Questa è una dictionary comprehension: un ciclo compatto che costruisce un dizionario.
    # Per ogni nome m di metrica, concateniamo 'test_' e m per trovare il relativo vettore in score.
    # Per esempio m='f1' diventa 'test_f1'; mean() calcola la media dei cinque valori.
    # Il risultato ha una chiave per metrica, per esempio {'f1': 0.70, ...}.
    # update aggiunge queste coppie chiave/valore al dizionario riga, che contiene già il nome della variante.
    riga.update({m: score['test_' + m].mean() for m in metriche_analisi})
    # std calcola la deviazione standard: descrive quanto F1 varia tra le cinque suddivisioni.
    # Una variazione piccola indica risultati più simili fra i fold di questa prova.
    # Non è l'errore di una singola previsione né un intervallo di confidenza del risultato.
    riga['std_f1'] = score['test_f1'].std()
    # append aggiunge la riga alla lista riassuntiva; modifica un contenitore esterno alla funzione.
    # A differenza del dizionario punteggi_analisi, la lista non sostituisce varianti con nomi uguali:
    # se richiamiamo la stessa prova più volte senza azzerarla, avremo righe ripetute.
    risultati_analisi.append(riga)
    # Mostriamo un riepilogo immediato: round(..., 4) arrotonda soltanto i numeri stampati.
    # La sottrazione tra i due contatori restituisce la durata della prova in secondi.
    # flush=True fa comparire subito il messaggio, senza aspettare che altre stampe si accumulino.
    print(nome, {m: round(riga[m], 4) for m in metriche_analisi},
          'secondi:', round(time.perf_counter() - inizio, 1), flush=True)
    # Restituiamo anche la sintesi al chiamante. Le celle successive possono ignorare questo valore
    # perché i risultati sono già stati inseriti nei due contenitori esterni.
    # La funzione conserva punteggi e riepiloghi: non restituisce i cinque modelli addestrati.
    return riga

# 11. CONTROLLI INIZIALI: PRIMA DEI PUNTEGGI, CAPIRE I DATI
# shape restituisce (numero di righe, numero di colonne); len conta gli indici riservati al test.
# A questo punto sono stati preparati dati e funzioni, ma non è ancora partita alcuna cross-validation.
print('CSV:', completo.shape, 'Campione:', campione.shape, 'Train:', train.shape,
      'Test riservato:', len(indici_test))
# value_counts conta quante risposte 0 e 1 sono presenti nel training.
# to_dict converte il risultato in un dizionario semplice da leggere nell'output.
print('Classi train:', y_train_analisi.value_counts().to_dict())
# isna restituisce True dove manca un valore; sum conta questi True per ogni colonna.
# loc conserva soltanto i conteggi maggiori di zero. lambda x: x > 0 è una piccola funzione
# senza nome: riceve la sequenza dei conteggi e dice quali tenere.
# Questo controllo è descrittivo sul CSV completo; non calcola valori per riempire il training.
print('Mancanti CSV:', completo.isna().sum().loc[lambda x: x > 0].to_dict())
# duplicated marca le righe identiche a una riga già incontrata, considerando tutte le colonne.
# sum conta le ripetizioni oltre la prima occorrenza: non è il numero totale di righe coinvolte.
# int rende il conteggio un intero Python. Senza un identificativo non sappiamo se si tratti
# di duplicazioni tecniche o di prenotazioni distinte con gli stessi valori.
print('Duplicati completi CSV:', int(completo.duplicated().sum()))
print('Duplicati completi campione:', int(campione.duplicated().sum()))
# Ribadiamo il criterio prima delle prove: confrontare F1 della classe cancellata, osservando anche gli altri punteggi.
# Le celle seguenti cambieranno le colonne, mentre campione, split, fold e impostazioni della foresta resteranno fissi.
print('La selezione usa F1 classe 1, con controllo di precision e recall. Nessuna valutazione sul test.')

CSV: (119390, 32) Campione: (12000, 32) Train: (9600, 32) Test riservato: 2400
Classi train: {0: 5977, 1: 3623}
Mancanti CSV: {'children': 4, 'country': 488, 'agent': 16340, 'company': 112593}
Duplicati completi CSV: 31994
Duplicati completi campione: 1551
La selezione usa F1 classe 1, con controllo di precision e recall. Nessuna valutazione sul test.


In [6]:
# ESPERIMENTO 1: capire il contributo di una colonna alla volta.
# La baseline, cioè il punto di partenza, usa numeriche_base e categoriche_base.
# RF abbrevia Random Forest. Ogni prova mantiene gli stessi dati, fold e impostazioni del modello.
# Cambia soltanto la colonna aggiunta, così il confronto resta interpretabile.
#
# La variabile aggiunte è una lista: le parentesi quadre esterne raccolgono le diverse prove.
# Ogni elemento è una tupla di tre valori: (nome della prova, nuove numeriche, nuove categoriche).
# Per esempio ('+ adr', ['adr'], []) aggiunge adr alle numeriche e nulla alle categoriche.
# Una lista vuota [] significa che non aggiungiamo colonne a quel gruppo.
#
# Esegui prima la cella di preparazione. valuta aggiunge risultati a risultati_analisi:
# rilanciare questa cella senza reinizializzare quella lista accumulerebbe righe duplicate.
# Per ripetere l'intero confronto da zero, esegui le celle dall'inizio del notebook.
aggiunte = [
    # La prima prova non aggiunge nulla: serve a misurare il riferimento con la stessa procedura delle altre.
    ('Base attuale RF', [], []),
    ('+ adr', ['adr'], []),
    # is_repeated_guest è un indicatore 0/1: viene trattato come una variabile numerica binaria.
    ('+ is_repeated_guest', ['is_repeated_guest'], []),
    ('+ distribution_channel', [], ['distribution_channel']),
    # agent e company sono codici identificativi, anche se nel CSV erano scritti come numeri.
    # Li aggiungiamo alle categorie: il codice 250 non significa 'più agenzia' del codice 10.
    ('+ agent categorico', [], ['agent']),
    ('+ company categorica', [], ['company']),
    ('+ total_of_special_requests', ['total_of_special_requests'], []),
    ('+ required_car_parking_spaces', ['required_car_parking_spaces'], []),
    ('+ deposit_type', [], ['deposit_type']),
]
# Il ciclo for visita una tupla alla volta. I suoi tre elementi vengono assegnati ai tre nomi:
# questa operazione si chiama spacchettamento (unpacking).
# Il simbolo + concatena liste di colonne: non somma i valori contenuti nelle prenotazioni.
# Per esempio numeriche_base + ['adr'] conserva le numeriche iniziali e aggiunge il nome adr.
for nome, extra_num, extra_cat in aggiunte:
    # Chiamiamo la funzione definita prima: costruisce una pipeline nuova e la valuta sui cinque fold.
    # Omettendo il quarto argomento, usiamo il classificatore predefinito, cioè la Random Forest.
    # Le medie finiscono nella lista risultati_analisi; i punteggi dei singoli fold in punteggi_analisi.
    valuta(nome, numeriche_base + extra_num, categoriche_base + extra_cat)
# Ora facciamo una prova di rimozione: lasciamo la baseline e togliamo una sola categoria.
# La scrittura [c for c in ... if ...] è una list comprehension, un modo compatto per creare una lista.
# Si legge: 'prendi ogni nome c da categoriche_base, tenendo solo quelli diversi da country'.
# Questo crea una nuova lista; categoriche_base rimane disponibile con tutte le colonne iniziali.
valuta('Senza country', numeriche_base, [c for c in categoriche_base if c != 'country'])
# Ripetiamo il ragionamento togliendo meal, sempre dalla baseline, senza togliere anche country.
valuta('Senza meal', numeriche_base, [c for c in categoriche_base if c != 'meal'])
# Trasformiamo la lista di dizionari in una tabella: ogni riga è una prova, ogni colonna una metrica.
# set_index usa il nome della variante come etichetta della riga al posto del numero progressivo.
confronto_analisi = pd.DataFrame(risultati_analisi).set_index('variante')
# Delta significa differenza. .loc seleziona un valore tramite le etichette di riga e colonna.
# Sottraiamo a ogni F1 medio quello della baseline: delta positivo = miglioramento medio;
# delta negativo = peggioramento medio; zero = stesso valore medio.
# Per esempio 0,75 - 0,70 = 0,05, cioè cinque punti sulla scala percentuale dell'F1,
# non un miglioramento relativo del 5%. Qui tutte le prove usano lo stesso tipo di modello.
confronto_analisi['delta_f1'] = confronto_analisi['f1'] - confronto_analisi.loc['Base attuale RF', 'f1']
# Mostriamo quattro decimali senza alterare i valori conservati in confronto_analisi.
# Sono medie di cross-validation sul training: il test finale non è stato usato.
# Una differenza piccola non dimostra da sola che una colonna sia sempre utile o dannosa:
# va considerata anche la variabilità tra fold e il contesto in cui il modello verrà usato.
display(confronto_analisi.round(4))

Base attuale RF {'accuracy': np.float64(0.8065), 'precision': np.float64(0.8421), 'recall': np.float64(0.5998), 'f1': np.float64(0.7003), 'roc_auc': np.float64(0.8881)} secondi: 4.2
+ adr {'accuracy': np.float64(0.8092), 'precision': np.float64(0.84), 'recall': np.float64(0.6108), 'f1': np.float64(0.7071), 'roc_auc': np.float64(0.8931)} secondi: 4.2
+ is_repeated_guest {'accuracy': np.float64(0.8035), 'precision': np.float64(0.8337), 'recall': np.float64(0.5989), 'f1': np.float64(0.6968), 'roc_auc': np.float64(0.8876)} secondi: 4.3
+ distribution_channel {'accuracy': np.float64(0.8036), 'precision': np.float64(0.8328), 'recall': np.float64(0.6003), 'f1': np.float64(0.6975), 'roc_auc': np.float64(0.8875)} secondi: 4.2
+ agent categorico {'accuracy': np.float64(0.806), 'precision': np.float64(0.8493), 'recall': np.float64(0.5907), 'f1': np.float64(0.6966), 'roc_auc': np.float64(0.9008)} secondi: 3.5
+ company categorica {'accuracy': np.float64(0.7983), 'precision': np.float64(0.8587), 'r

,accuracy,precision,recall,f1,roc_auc,std_f1,delta_f1
variante,,,,,,,
Base attuale RF,0.8065,0.8421,0.5998,0.7003,0.8881,0.0182,0.0000
+ adr,0.8092,0.8400,0.6108,0.7071,0.8931,0.0130,0.0068
+ is_repeated_guest,0.8035,0.8337,0.5989,0.6968,0.8876,0.0194,-0.0035
+ distribution_channel,0.8036,0.8328,0.6003,0.6975,0.8875,0.0147,-0.0027
+ agent categorico,0.8060,0.8493,0.5907,0.6966,0.9008,0.0203,-0.0037
+ company categorica,0.7983,0.8587,0.5575,0.6758,0.8862,0.0146,-0.0244
+ total_of_special_requests,0.8328,0.8442,0.6829,0.7548,0.9068,0.0205,0.0545
+ required_car_parking_spaces,0.8127,0.8374,0.6252,0.7156,0.8972,0.0179,0.0154
+ deposit_type,0.8052,0.8566,0.5810,0.6923,0.8902,0.0150,-0.0080


In [7]:
# ESPERIMENTO 2: verificare se le colonne promettenti funzionano bene anche insieme.
# Il loro contributo può sovrapporsi: due miglioramenti separati non si sommano automaticamente.
# Questa cella riusa le funzioni, i dati e i fold preparati prima; il test resta escluso.
# valuta continua ad aggiungere righe alla stessa lista risultati_analisi e salva i punteggi per nome.
# Rilanciare questa cella da sola accumulerebbe righe duplicate: per ripartire, esegui dall'inizio.
#
# Prima aggiungiamo richieste speciali e posti auto alla baseline della Random Forest.
# Nessun classificatore è passato come quarto argomento: vale quello predefinito nella funzione.
valuta('Richieste + parcheggio', numeriche_base + ['total_of_special_requests', 'required_car_parking_spaces'], categoriche_base)
# Questa seconda combinazione aggiunge anche adr, la tariffa media giornaliera.
# Confrontandola con la combinazione precedente valutiamo il contributo aggiuntivo di adr.
valuta('Richieste + parcheggio + adr', numeriche_base + ['total_of_special_requests', 'required_car_parking_spaces', 'adr'], categoriche_base)
# Controlliamo se il beneficio dipende dal classificatore: misuriamo una baseline logistica separata.
# Il quarto argomento sostituisce la Random Forest predefinita con LogisticRegression.
# max_iter=2000 limita i passi dell'ottimizzazione: non indica quante prenotazioni vengono usate.
valuta('Base attuale logistica', numeriche_base, categoriche_base, LogisticRegression(max_iter=2000))
# Ora ripetiamo le tre aggiunte con la regressione logistica, usando gli stessi cinque fold.
# Per misurare il beneficio delle colonne nella logistica dobbiamo confrontare questa prova
# con 'Base attuale logistica', che mantiene lo stesso classificatore e cambia solo le feature.
valuta('Tre aggiunte logistica', numeriche_base + ['total_of_special_requests', 'required_car_parking_spaces', 'adr'],
       categoriche_base, LogisticRegression(max_iter=2000))
# Ricreiamo la tabella dalla lista aggiornata: include sia gli esperimenti precedenti sia questi.
confronto_analisi = pd.DataFrame(risultati_analisi).set_index('variante')
# Attenzione al riferimento di questa colonna: sottrae SEMPRE l'F1 della Random Forest di base,
# anche nelle righe della regressione logistica. Perciò nelle righe logistiche la differenza
# include il cambio di classificatore e non isola da sola l'effetto delle colonne aggiunte.
# Il confronto con la base dello STESSO modello viene invece calcolato per ogni fold qui sotto.
confronto_analisi['delta_f1_RF_base'] = confronto_analisi['f1'] - confronto_analisi.loc['Base attuale RF', 'f1']
# L'arrotondamento serve solo a rendere leggibile la tabella delle medie di cross-validation.
display(confronto_analisi.round(4))
# Finora abbiamo confrontato le medie; adesso guardiamo le cinque differenze una per una.
# Poiché ogni modello usa gli stessi fold, il primo punteggio di una variante è confrontabile
# con il primo della sua baseline, il secondo con il secondo, e così via.
print('Delta F1 per fold, rispetto alla base dello stesso modello:')
# La lista contiene coppie (variante, riferimento corretto). Il ciclo spacchetta ogni coppia
# nei nomi nome e nome_base: le varianti RF usano la base RF, la logistica usa la base logistica.
for nome, nome_base in [('Richieste + parcheggio', 'Base attuale RF'),
                        ('Richieste + parcheggio + adr', 'Base attuale RF'),
                        ('Tre aggiunte logistica', 'Base attuale logistica')]:
    # punteggi_analisi[nome] recupera il dizionario dei risultati di quella variante.
    # ['test_f1'] estrae l'array con i cinque F1 ottenuti nei fold di validazione.
    # Qui la parola test nel nome della chiave NON indica il test finale riservato all'inizio.
    # NumPy sottrae i due array elemento per elemento, ottenendo cinque differenze appaiate.
    # np.round(..., 4) arrotonda i numeri; .tolist() li converte in una normale lista Python da stampare.
    print(nome, np.round(punteggi_analisi[nome]['test_f1'] -
                         punteggi_analisi[nome_base]['test_f1'], 4).tolist())
# Cinque differenze positive indicano un miglioramento osservato in ciascuna di queste suddivisioni.
# Non costituiscono da sole una prova statistica definitiva né garantiscono lo stesso vantaggio su dati futuri.
# Stiamo ancora scegliendo con la validazione del training, senza misurare queste varianti sul test finale.
print('Non sono stati calcolati nuovi risultati sul test.')

Richieste + parcheggio {'accuracy': np.float64(0.8379), 'precision': np.float64(0.8459), 'recall': np.float64(0.6975), 'f1': np.float64(0.7644), 'roc_auc': np.float64(0.9149)} secondi: 4.2
Richieste + parcheggio + adr {'accuracy': np.float64(0.8418), 'precision': np.float64(0.8573), 'recall': np.float64(0.6967), 'f1': np.float64(0.7685), 'roc_auc': np.float64(0.9181)} secondi: 4.2
Base attuale logistica {'accuracy': np.float64(0.7742), 'precision': np.float64(0.7535), 'recall': np.float64(0.5965), 'f1': np.float64(0.6658), 'roc_auc': np.float64(0.8386)} secondi: 0.5
Tre aggiunte logistica {'accuracy': np.float64(0.8046), 'precision': np.float64(0.7739), 'recall': np.float64(0.6812), 'f1': np.float64(0.7244), 'roc_auc': np.float64(0.8787)} secondi: 0.5


,accuracy,precision,recall,f1,roc_auc,std_f1,delta_f1_RF_base
variante,,,,,,,
Base attuale RF,0.8065,0.8421,0.5998,0.7003,0.8881,0.0182,0.0000
+ adr,0.8092,0.8400,0.6108,0.7071,0.8931,0.0130,0.0068
+ is_repeated_guest,0.8035,0.8337,0.5989,0.6968,0.8876,0.0194,-0.0035
+ distribution_channel,0.8036,0.8328,0.6003,0.6975,0.8875,0.0147,-0.0027
+ agent categorico,0.8060,0.8493,0.5907,0.6966,0.9008,0.0203,-0.0037
+ company categorica,0.7983,0.8587,0.5575,0.6758,0.8862,0.0146,-0.0244
+ total_of_special_requests,0.8328,0.8442,0.6829,0.7548,0.9068,0.0205,0.0545
+ required_car_parking_spaces,0.8127,0.8374,0.6252,0.7156,0.8972,0.0179,0.0154
+ deposit_type,0.8052,0.8566,0.5810,0.6923,0.8902,0.0150,-0.0080


Delta F1 per fold, rispetto alla base dello stesso modello:
Richieste + parcheggio [0.0928, 0.074, 0.0551, 0.0398, 0.0589]
Richieste + parcheggio + adr [0.0962, 0.0696, 0.0579, 0.0452, 0.0724]
Tre aggiunte logistica [0.0894, 0.0647, 0.0506, 0.0452, 0.0433]
Non sono stati calcolati nuovi risultati sul test.


In [8]:
# ESPERIMENTO 3: controllare se il miglioramento resiste quando teniamo insieme i profili uguali.
# Una suddivisione casuale può mettere due righe identiche una nel training e una nella validazione.
# Questo può rendere la valutazione più ottimistica: il modello incontra un caso uguale a uno già visto.
# Raggruppiamo quindi i profili coincidenti e assegniamo ogni gruppo a un solo fold.
# Non sappiamo se quelle righe siano duplicazioni tecniche o prenotazioni distinte con dati uguali:
# un gruppo qui identifica un profilo osservato, non un cliente o una prenotazione certa.
# Controllo di robustezza: righe con lo stesso profilo non attraversano i fold.
# Non si eliminano prenotazioni: si mantengono insieme i profili coincidenti.
# StratifiedGroupKFold rispetta i gruppi e cerca contemporaneamente di mantenere
# proporzioni simili di cancellazioni nei fold. Con gruppi indivisibili l'equilibrio è approssimato.
from sklearn.model_selection import StratifiedGroupKFold
# Creiamo la lista di colonne con cui descrivere un profilo.
# La list comprehension visita i nomi c in train.columns e scarta quelli elencati sotto.
# Escludiamo la risposta is_canceled e lo stato finale con la sua data: il criterio di uguaglianza
# non deve dipendere dall'esito che vogliamo prevedere.
# Le colonne usate per definire i gruppi non diventano per questo ingressi del classificatore:
# il modello continua a ricevere soltanto numeriche e categoriche selezionate.
colonne_gruppo = [c for c in train.columns if c not in
                  ['is_canceled', 'reservation_status', 'reservation_status_date']]
# Un hash è una firma numerica calcolata dal contenuto della riga: profili uguali danno la stessa firma.
# index=False esclude dall'impronta l'etichetta di riga, così due profili uguali restano nello stesso gruppo
# anche se occupano posizioni diverse. Conserviamo una firma per ogni riga del training.
# Il raggruppamento non elimina righe e non usa queste firme come caratteristica per prevedere.
gruppi = pd.util.hash_pandas_object(train[colonne_gruppo], index=False)
# Prepariamo cinque fold con vincolo sui gruppi. shuffle introduce casualità nella procedura
# e random_state la rende riproducibile sugli stessi dati e con la stessa implementazione.
# split restituisce, per ogni fold, una coppia di array: posizioni per addestrare e posizioni per validare.
# list conserva tutte le coppie, che saranno riutilizzate identiche per entrambi i modelli.
fold_gruppi = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
                  .split(train, y_train_analisi, groups=gruppi))
# Usiamo una lista separata per non mescolare questa procedura con i precedenti fold casuali.
# Questa assegnazione azzera la lista ogni volta che si esegue la cella.
risultati_gruppi = []
# Confrontiamo due configurazioni della stessa Random Forest: colonne di base e tre aggiunte.
# Ogni elemento della lista è una coppia (nome, elenco numeriche); il for assegna i due valori a nome e num.
# Le categoriche rimangono le stesse, così il confronto isola la modifica delle tre numeriche.
for nome, num in [('RF base con gruppi', numeriche_base),
                  ('RF tre aggiunte con gruppi', numeriche_base +
                   ['total_of_special_requests', 'required_car_parking_spaces', 'adr'])]:
    # crea_modello costruisce una pipeline nuova; cross_validate la riaddestra nei cinque fold per gruppi.
    # train[num + categoriche_base] seleziona soltanto gli ingressi del modello; y_train_analisi è la risposta reale.
    # cv=fold_gruppi applica le suddivisioni appena create; scoring richiede le stesse metriche di prima.
    # n_jobs=1 elabora un fold alla volta; error_score='raise' rende visibile un eventuale errore.
    score = cross_validate(crea_modello(num, categoriche_base),
                           train[num + categoriche_base], y_train_analisi,
                           cv=fold_gruppi, scoring=metriche_analisi, n_jobs=1, error_score='raise')
    # Prepariamo un dizionario per raccogliere il nome della configurazione e le sue medie.
    riga = {'variante': nome}
    # Questa è una dictionary comprehension: per ogni nome di metrica m costruiamo una coppia chiave/valore.
    # Per esempio m='f1' cerca score['test_f1'] e ne calcola la media sui cinque fold.
    # update aggiunge tutte queste coppie al dizionario riga. I punteggi test_... sono di validazione,
    # non del test finale tenuto da parte nella prima cella.
    riga.update({m: score['test_' + m].mean() for m in metriche_analisi})
    # La deviazione standard riassume quanto variano i cinque F1: non è l'errore di una singola previsione
    # e non è, da sola, un intervallo di confidenza sul valore medio.
    riga['std_f1'] = score['test_f1'].std()
    # append aggiunge il dizionario come nuova riga nella lista dei risultati.
    risultati_gruppi.append(riga)
    # Mostriamo le metriche con quattro decimali. flush=True fa apparire subito il messaggio
    # per seguire l'avanzamento senza aspettare che tutte le valutazioni siano terminate.
    print(nome, {m: round(riga[m], 4) for m in metriche_analisi}, flush=True)
# Costruiamo e mostriamo la tabella finale di questo controllo.
# Confrontiamo prima le due righe fra loro: condividono gli stessi fold per gruppi.
# Le medie dei precedenti fold casuali descrivono una procedura diversa, quindi non sono lo stesso esperimento.
display(pd.DataFrame(risultati_gruppi).set_index('variante').round(4))
# Verifichiamo esplicitamente che nessun gruppo attraversi training e validazione dello stesso fold.
# nunique conta quante firme distinte esistono, cioè quanti profili raggruppati ci sono.
# Nella lista sotto, it e iv sono gli array di posizioni delle righe di training e validazione.
# .iloc seleziona per posizione: serve perché split restituisce posizioni, non etichette dell'indice.
# set elimina firme ripetute; il simbolo & calcola l'intersezione, cioè le firme comuni ai due insiemi.
# len conta quante firme sono condivise; la list comprehension ripete il controllo sui cinque fold.
# Se il vincolo sui gruppi è rispettato, il risultato atteso è [0, 0, 0, 0, 0].
print('Gruppi distinti nel train:', gruppi.nunique(),
      '- sovrapposizione gruppi tra training/validazione:',
      [len(set(gruppi.iloc[it]) & set(gruppi.iloc[iv])) for it, iv in fold_gruppi])
# Questo controllo evita la sovrapposizione dei profili definiti qui; non può garantire
# che lo stesso cliente o la stessa prenotazione, con valori diversi, non compaiano in più fold.
print('I gruppi sono profili coincidenti, non identificativi certi di prenotazioni o clienti.')
# Tenere insieme i profili uguali non verifica se il modello sa prevedere prenotazioni future.
# Per quello servirebbe separare i dati nel tempo, addestrando sul passato e valutando sul periodo successivo.
# Inoltre, le colonne devono essere già disponibili quando si vuole fare la previsione:
# il raggruppamento non corregge eventuali informazioni aggiornate dopo quel momento.
print('Lo split temporale richiederebbe una valutazione separata; il test resta escluso.')

RF base con gruppi {'accuracy': np.float64(0.7931), 'precision': np.float64(0.8177), 'recall': np.float64(0.5816), 'f1': np.float64(0.6797), 'roc_auc': np.float64(0.8766)}
RF tre aggiunte con gruppi {'accuracy': np.float64(0.836), 'precision': np.float64(0.851), 'recall': np.float64(0.6856), 'f1': np.float64(0.7594), 'roc_auc': np.float64(0.9107)}


,accuracy,precision,recall,f1,roc_auc,std_f1
variante,,,,,,
RF base con gruppi,0.7931,0.8177,0.5816,0.6797,0.8766,0.0113
RF tre aggiunte con gruppi,0.8360,0.8510,0.6856,0.7594,0.9107,0.0080


Gruppi distinti nel train: 8476 - sovrapposizione gruppi tra training/validazione: [0, 0, 0, 0, 0]
I gruppi sono profili coincidenti, non identificativi certi di prenotazioni o clienti.
Lo split temporale richiederebbe una valutazione separata; il test resta escluso.


## Valutazione e modifica suggerita
La scelta iniziale dei tipi e corretta: durate/conteggi nelle numeriche, etichette nelle categoriche (compreso il mese). La configurazione migliore fra quelle provate conserva tutte le colonne iniziali e aggiunge tre numeriche: total_of_special_requests, required_car_parking_spaces e adr.
- Random Forest: F1 CV da 0,7003 a 0,7685; recall da 59,98% a 69,67%; precision da 84,21% a 85,73%; accuracy da 80,65% a 84,18%.
- Regressione logistica: F1 CV da 0,6658 a 0,7244; recall da 59,65% a 68,12%.
Il miglioramento della combinazione e positivo in tutti e cinque i fold, per entrambi i modelli. La maggior parte del beneficio viene dalle richieste speciali; aggiungere adr alle richieste piu parcheggio porta un margine piccolo (F1 0,7644 -> 0,7685).
Mantenere country: senza questa colonna F1 RF scende a 0,6357. Non eliminare meal sulla base di questo confronto: senza, F1 e 0,6945. Non aggiungere per ora is_repeated_guest, distribution_channel, company o deposit_type: aggiunte singolarmente non migliorano F1 RF. Agent migliora AUC (0,8881 -> 0,9008) ma non F1 alla soglia standard (0,7003 -> 0,6966); puo essere una futura prova con soglia scelta in CV. Le piccole differenze non dimostrano un peggioramento generale.
Se si provano agent/company, sono identificativi categorici, con NULL come categoria non applicabile (non media o moda); company ha 94,31% di valori non applicabili.
## Limiti di interpretazione
Il paper originale descrive dati ricostruiti al giorno precedente l'arrivo: non e garantito che siano i valori alla creazione della prenotazione. Richieste, parcheggio e ADR vanno inclusi in una previsione operativa solo se gia disponibili a quel momento; lo stesso controllo vale per country. DepositType usa pagamenti effettivi registrati prima di arrivo/cancellazione, non soltanto una politica contrattuale.
Escludere reservation_status e reservation_status_date: descrivono l'esito. Per prevedere alla creazione, escludere anche booking_changes e assigned_room_type se aggiornati successivamente; days_in_waiting_list presuppone la conferma.
Ci sono 31.994 righe complete ripetute nel CSV, di cui 1.551 nel campione: senza identificativi non si puo stabilire se siano duplicazioni tecniche o prenotazioni distinte con dati uguali. Non rimuoverle automaticamente. Nel controllo aggiuntivo con StratifiedGroupKFold, mantenendo nello stesso fold i profili identici (senza target e stati finali nella chiave), F1 RF passa da 0,6797 a 0,7594, recall da 58,16% a 68,56%. Non c'e sovrapposizione dei gruppi fra training e validazione. Il miglioramento persiste; la validazione temporale rimane una verifica distinta per un uso futuro.
Fonte primaria: https://repositorio.iscte-iul.pt/bitstream/10071/16929/1/Hotelbookingdemanddatasets.pdf
Il test originale non e stato riutilizzato in questa selezione. Il notebook originale e invariato.
## Codice da usare nel notebook originale
Sostituire le due liste nella cella di preprocessing con quelle sotto, prima di costruire X e della divisione train/test. Si possono mantenere le pipeline attuali.

In [9]:
# SINTESI PRATICA: QUALI NOMI DI COLONNA PORTARE NELL'ESERCIZIO
# Queste liste descrivono la configurazione suggerita dal confronto; non eseguono un nuovo addestramento.
# I nomi terminano con _suggerite per non sovrascrivere numeriche_base e categoriche_base:
# la baseline deve restare la stessa quando confrontiamo le varianti.
# Usare queste caratteristiche presuppone che i valori siano già disponibili quando si vuole prevedere.
# Colonne suggerite per l'esercizio, se disponibili al momento della previsione.
# Conserviamo le otto numeriche di partenza e aggiungiamo i tre conteggi/importi valutati insieme.
# Ogni elemento è una stringa, cioè un testo che deve coincidere con un nome di colonna nel CSV.
numeriche_suggerite = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies',
    'previous_cancellations', 'previous_bookings_not_canceled',
    # Le tre aggiunte sono: numero di richieste speciali, numero di posti auto richiesti e tariffa media giornaliera.
    'total_of_special_requests', 'required_car_parking_spaces', 'adr',
]
# Le sette categorie di partenza vengono mantenute; il confronto non suggerisce qui di cambiarne il tipo.
# Questi testi dovranno essere trasformati in numeri tramite l'encoder già presente nella pipeline.
categoriche_suggerite = [
    'hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
    'reserved_room_type', 'customer_type',
]
# La sola definizione di una lista non aggiorna un modello già costruito o addestrato.
# Per usare la proposta in esercizio_hotel.ipynb occorre sostituire le sue liste, poi rieseguire
# la costruzione di X, lo split, il preprocessing e gli addestramenti che dipendono da quelle liste.
# I risultati numerici scritti nel riepilogo si riferiscono agli esperimenti eseguiti qui.
# Nel notebook originale usare i nomi numeriche e categoriche,
# poi ricostruire X, split, preprocessing e modelli.